In [4]:
from ultralytics import YOLO
import os
from pathlib import Path
import torch
import random
import shutil

In [52]:
# Clean data
data_folder = 'a'

# Read the lines from the file
traintxt = os.path.join(data_folder, "train.txt")
with open(traintxt, 'r') as f:
    train_lines = f.read().splitlines()

all_filenames = set([Path(line).stem for line in train_lines])

# Find missing
found_filenames = set()
for filename in os.listdir(os.path.join(data_folder, "labels/train")):
    filename_stem = Path(filename).stem
    found_filenames.add(filename_stem)

missing_filenames = all_filenames - found_filenames 
print(f"{len(all_filenames)} total images")
print(f"{len(found_filenames)} labels found")
print(f"Removing {len(missing_filenames)} images")

# Remove the missing images
images_path = os.path.join(data_folder, "images/train")
for filename in os.listdir(images_path):
    filename_stem = Path(filename).stem
    if filename_stem in missing_filenames:
        os.remove(os.path.join(images_path, filename))

# Re-save the train.txt with only real images
with open(traintxt, "w") as f:
    f.write('\n'.join(found_filenames))


1461 total images
283 labels found
Removing 1178 images


In [65]:
# Do train/val split
assert found_filenames
TEST_SPLIT = 0.1

test_filenames = set(random.sample(sorted(found_filenames), int(len(found_filenames) * TEST_SPLIT)))
train_filenames = found_filenames - test_filenames

# Create folders and move files
os.mkdir(os.path.join(data_folder, 'images/val'))
for filename in test_filenames:
    org_path = os.path.join(data_folder, "images/train/", f"{filename}.jpg")
    new_path = os.path.join(data_folder, "images/val/", f"{filename}.jpg")
    shutil.move(org_path, new_path)

os.mkdir(os.path.join(data_folder, 'labels/val'))
for filename in test_filenames:
    org_path = os.path.join(data_folder, "labels/train/", f"{filename}.txt")
    new_path = os.path.join(data_folder, "labels/val/", f"{filename}.txt")
    shutil.move(org_path, new_path)

# Update train.txt, add val.txt and update data.yaml
with open(os.path.join(data_folder, 'train.txt'), 'w') as f:
    f.write('\n'.join(f"./images/train/{f}.jpg" for f in train_filenames))
    
with open(os.path.join(data_folder, 'val.txt'), 'w') as f:
    f.write('\n'.join(f"./images/val/{f}.jpg" for f in test_filenames))

with open(os.path.join(data_folder, 'data.yaml'), 'a') as f:
    f.write("val: val.txt\n")


FileExistsError: [Errno 17] File exists: 'a/images/val'

In [6]:
# Train yolo
model = YOLO("yolo26s.pt")

if torch.accelerator.is_available():
    torch.device(torch.accelerator.current_accelerator)
else:
    torch.device('cpu')

results = model.train(data="./a/data.yaml", epochs=1, batch=-1, resume=True, device="cpu")

New https://pypi.org/project/ultralytics/8.4.40 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.26 🚀 Python-3.14.0 torch-2.11.0+cu130 CPU (AMD Ryzen 7 5800H with Radeon Graphics)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=9.83241, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.64896, cls_w=3.48357, compile=False, conf=None, copy_paste=0.40413, copy_paste_mode=flip, cos_lr=False, cutmix=0.00082, data=./a/data.yaml, degrees=0.00012, detach_epoch=10, deterministic=True, device=cpu, dfl=0.95824, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=True, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.30393, flipud=0.00653, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01315, hsv_s=0.35348, hsv_v=0.19383, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00038, lrf=0.88219, mask_ratio=4, max_det=300, mixup=0.05, mode=train, m

SyntaxError: '[31m[1mmuon_w[0m' is not a valid YOLO argument. 
'[31m[1mcls_w[0m' is not a valid YOLO argument. Similar arguments are i.e. ['cls=0.5'].
'[31m[1mtopk[0m' is not a valid YOLO argument. 
'[31m[1msgd_w[0m' is not a valid YOLO argument. 
'[31m[1mstride_ratio[0m' is not a valid YOLO argument. Similar arguments are i.e. ['mask_ratio=4'].
'[31m[1mdetach_epoch[0m' is not a valid YOLO argument. 
'[31m[1mo2m[0m' is not a valid YOLO argument. 

    Arguments received: ['yolo', '-f', '/home/tino/.local/share/jupyter/runtime/kernel-1888781f-e922-4d8c-aaec-fe66de1e774c.json']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['detect', 'segment', 'pose', 'obb', 'classify']
                MODE (required) is one of ['export', 'track', 'benchmark', 'val', 'train', 'predict']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
  